In [3]:
import numpy as np

def bpx_scaled_1d(L):
    n = 2**L - 1                  # fine-grid interior nodes
    cols = sum(2**l - 1 for l in range(1, L+1))
    F = np.zeros((n, cols), dtype=float)

    curr_col = 0
    for ell in range(1, L+1):
        m = 2**(L - ell)          # fine-grid steps to neighbor coarse node
        mult = 2.0**(-ell/2.0)    # BPX scaling for d=1
        peak = m - 1              # first interior coarse node index on fine grid
        for j in range(1, 2**ell):  # there are 2^ell - 1 interior coarse nodes
            f = np.zeros(n, dtype=float)
            # center
            f[peak] = mult
            # linear decay to neighbors over m-1 interior steps
            for val in range(1, m):
                w = mult * (1.0 - val / m)
                f[peak + val] = w
                f[peak - val] = w
            F[:, curr_col] = f
            curr_col += 1
            peak += m
    return F

In [1]:
def print_matrix_rounded(L, precision=3):
    # Round the entries
    L_rounded = np.round(L, precision)
    # Print each row
    for row in L_rounded:
        print("[ " + "  ".join(f"{val:.{precision}f}" for val in row) + " ]")

In [4]:
def S_1d(L):
    h = 1.0 / (2**L)
    n = 2**L - 1
    e = np.ones(n)
    T = np.diag(2*e) - np.diag(e[:-1],1) - np.diag(e[:-1],-1)
    return (1.0/h)* (1.0/h)* T

for L in range(3, 12):
    F = bpx_scaled_1d(L)
    S = S_1d(L)
    A = F.T @ S @ F
    w = np.linalg.eigvalsh(A)
    w.sort()
    w = w[::-1]
    print(L, w[0]/w[2**L - 2])

3 2.8660254037844375
4 3.4838540165886243
5 3.9833604689984328
6 4.389073300164275
7 4.723936471498686
8 5.002949323913253
9 5.237564954564484
10 5.436435694138228
11 5.606254052125858


In [6]:
print_matrix_rounded(bpx_scaled_1d(3))

[ 0.177  0.250  0.000  0.000  0.354  0.000  0.000  0.000  0.000  0.000  0.000 ]
[ 0.354  0.500  0.000  0.000  0.000  0.354  0.000  0.000  0.000  0.000  0.000 ]
[ 0.530  0.250  0.250  0.000  0.000  0.000  0.354  0.000  0.000  0.000  0.000 ]
[ 0.707  0.000  0.500  0.000  0.000  0.000  0.000  0.354  0.000  0.000  0.000 ]
[ 0.530  0.000  0.250  0.250  0.000  0.000  0.000  0.000  0.354  0.000  0.000 ]
[ 0.354  0.000  0.000  0.500  0.000  0.000  0.000  0.000  0.000  0.354  0.000 ]
[ 0.177  0.000  0.000  0.250  0.000  0.000  0.000  0.000  0.000  0.000  0.354 ]


In [7]:
def Forward(n):
    F = np.zeros((n, n+1))
    h = 1.0/(n+1)
    for i in range(n):
        F[i, i] = 1.0/h
        F[i, i+1] = -1.0/h
    return F

In [8]:
print_matrix_rounded(S_1d(3))

[ 128.000  -64.000  0.000  0.000  0.000  0.000  0.000 ]
[ -64.000  128.000  -64.000  0.000  0.000  0.000  0.000 ]
[ 0.000  -64.000  128.000  -64.000  0.000  0.000  0.000 ]
[ 0.000  0.000  -64.000  128.000  -64.000  0.000  0.000 ]
[ 0.000  0.000  0.000  -64.000  128.000  -64.000  0.000 ]
[ 0.000  0.000  0.000  0.000  -64.000  128.000  -64.000 ]
[ 0.000  0.000  0.000  0.000  0.000  -64.000  128.000 ]


In [11]:
np.sort(np.linalg.eigvals(S_1d(3)))

array([  9.74341984,  37.49033201,  79.01652066, 128.        ,
       176.98347934, 218.50966799, 246.25658016])

In [14]:
S = S_1d(3)
B = bpx_scaled_1d(3)
K = B.T @ S @ B

In [15]:
print_matrix_rounded(K, precision=2)

[ 16.00  0.00  11.31  0.00  0.00  0.00  0.00  8.00  0.00  -0.00  0.00 ]
[ 0.00  16.00  -8.00  0.00  0.00  11.31  0.00  -5.66  0.00  0.00  0.00 ]
[ 11.31  -8.00  16.00  -8.00  0.00  -5.66  0.00  11.31  0.00  -5.66  0.00 ]
[ 0.00  0.00  -8.00  16.00  0.00  0.00  0.00  -5.66  0.00  11.31  0.00 ]
[ 0.00  0.00  0.00  0.00  16.00  -8.00  0.00  0.00  0.00  0.00  0.00 ]
[ 0.00  11.31  -5.66  0.00  -8.00  16.00  -8.00  0.00  0.00  0.00  0.00 ]
[ 0.00  0.00  0.00  0.00  0.00  -8.00  16.00  -8.00  0.00  0.00  0.00 ]
[ 8.00  -5.66  11.31  -5.66  0.00  0.00  -8.00  16.00  -8.00  0.00  0.00 ]
[ 0.00  0.00  0.00  0.00  0.00  0.00  0.00  -8.00  16.00  -8.00  0.00 ]
[ 0.00  0.00  -5.66  11.31  0.00  0.00  0.00  0.00  -8.00  16.00  -8.00 ]
[ 0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  -8.00  16.00 ]


In [16]:
np.sort(np.linalg.eigvals(K))

array([-1.61562684e-15, -8.68787407e-16,  0.00000000e+00,  3.09167054e-15,
        1.60000000e+01,  1.60000000e+01,  1.60000000e+01,  1.81435935e+01,
        3.20000000e+01,  3.20000000e+01,  4.58564065e+01])

In [17]:
for L in range(3, 12):
    F = bpx_scaled_1d(L)
    S = S_1d(L)
    A =  S
    w = np.linalg.eigvalsh(A)
    w.sort()
    w = w[::-1]
    print(L, w[0]/w[2**L - 2])

3 25.2741423690881
4 103.08686891981701
5 414.3450622319025
6 1659.3796462926853
7 6639.518434559989
8 26560.073700500692
9 106242.29479210546
10 424971.17918511067
11 1699886.7204808758


In [18]:
For = Forward(7)

In [19]:
print_matrix_rounded(For)

[ 8.000  -8.000  0.000  0.000  0.000  0.000  0.000  0.000 ]
[ 0.000  8.000  -8.000  0.000  0.000  0.000  0.000  0.000 ]
[ 0.000  0.000  8.000  -8.000  0.000  0.000  0.000  0.000 ]
[ 0.000  0.000  0.000  8.000  -8.000  0.000  0.000  0.000 ]
[ 0.000  0.000  0.000  0.000  8.000  -8.000  0.000  0.000 ]
[ 0.000  0.000  0.000  0.000  0.000  8.000  -8.000  0.000 ]
[ 0.000  0.000  0.000  0.000  0.000  0.000  8.000  -8.000 ]


In [20]:
print_matrix_rounded(For @ For.T, precision=2)

[ 128.00  -64.00  0.00  0.00  0.00  0.00  0.00 ]
[ -64.00  128.00  -64.00  0.00  0.00  0.00  0.00 ]
[ 0.00  -64.00  128.00  -64.00  0.00  0.00  0.00 ]
[ 0.00  0.00  -64.00  128.00  -64.00  0.00  0.00 ]
[ 0.00  0.00  0.00  -64.00  128.00  -64.00  0.00 ]
[ 0.00  0.00  0.00  0.00  -64.00  128.00  -64.00 ]
[ 0.00  0.00  0.00  0.00  0.00  -64.00  128.00 ]


In [21]:
# B.T @ L @ B = B.T @ F @ F.T @ B
# <y|B.T @ F @ F.T @ B|y> 
# = || F.T @ B |y> ||^2 


# somehow this has to differ from

# L = F @ F.T
# ||F.T |y> ||^2

In [ ]:
x = np.ones((7, 1))
y = np.zeros((7, 1)); y[0] = 1.0

In [ ]:
print_matrix_rounded(For.T @ x)
# \sqrt(8^2*2)/ \sqrt(1^2*7)

[ 8.000 ]
[ 0.000 ]
[ 0.000 ]
[ 0.000 ]
[ 0.000 ]
[ 0.000 ]
[ 0.000 ]
[ -8.000 ]


In [ ]:
print_matrix_rounded(For.T @ y)
# \sqrt(8^2*2)/ \sqrt(1^2)

[ 8.000 ]
[ -8.000 ]
[ 0.000 ]
[ 0.000 ]
[ 0.000 ]
[ 0.000 ]
[ 0.000 ]
[ 0.000 ]


In [29]:
print_matrix_rounded(For.T @ B)

[ 1.414  2.000  0.000  0.000  2.828  0.000  0.000  0.000  0.000  0.000  0.000 ]
[ 1.414  2.000  0.000  0.000  -2.828  2.828  0.000  0.000  0.000  0.000  0.000 ]
[ 1.414  -2.000  2.000  0.000  0.000  -2.828  2.828  0.000  0.000  0.000  0.000 ]
[ 1.414  -2.000  2.000  0.000  0.000  0.000  -2.828  2.828  0.000  0.000  0.000 ]
[ -1.414  0.000  -2.000  2.000  0.000  0.000  0.000  -2.828  2.828  0.000  0.000 ]
[ -1.414  0.000  -2.000  2.000  0.000  0.000  0.000  0.000  -2.828  2.828  0.000 ]
[ -1.414  0.000  0.000  -2.000  0.000  0.000  0.000  0.000  0.000  -2.828  2.828 ]
[ -1.414  0.000  0.000  -2.000  0.000  0.000  0.000  0.000  0.000  0.000  -2.828 ]


In [30]:
print_matrix_rounded(B.T @ For @ For.T @ B, precision=2)

[ 16.00  0.00  11.31  0.00  0.00  -0.00  0.00  8.00  0.00  -0.00  0.00 ]
[ 0.00  16.00  -8.00  0.00  0.00  11.31  0.00  -5.66  0.00  0.00  0.00 ]
[ 11.31  -8.00  16.00  -8.00  0.00  -5.66  0.00  11.31  0.00  -5.66  0.00 ]
[ 0.00  0.00  -8.00  16.00  0.00  0.00  0.00  -5.66  0.00  11.31  0.00 ]
[ 0.00  0.00  0.00  0.00  16.00  -8.00  0.00  0.00  0.00  0.00  0.00 ]
[ 0.00  11.31  -5.66  0.00  -8.00  16.00  -8.00  0.00  0.00  0.00  0.00 ]
[ 0.00  0.00  0.00  0.00  0.00  -8.00  16.00  -8.00  0.00  0.00  0.00 ]
[ 8.00  -5.66  11.31  -5.66  0.00  0.00  -8.00  16.00  -8.00  0.00  0.00 ]
[ 0.00  0.00  0.00  0.00  0.00  0.00  0.00  -8.00  16.00  -8.00  0.00 ]
[ 0.00  0.00  -5.66  11.31  0.00  0.00  0.00  0.00  -8.00  16.00  -8.00 ]
[ 0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  -8.00  16.00 ]
